In [1]:
from cresowlve.utils import read_json

model_results_path = "../experiments/outputs/gemini-2.5-pro/chgk_en_benchmark_eval_reasoning_model_en_s0_gemini-2.5-pro_20260228_163314.json"
reasoning_types_path = "../experiments/data/task/chgk_benchmark_reasoning_types.json"
benchmark_path = "../experiments/data/task/chgk_en_benchmark.json"
model_results = read_json(model_results_path)
reasoning_types_results = read_json(reasoning_types_path)
benchmark = read_json(benchmark_path)

for sample in model_results["data"]:
    benchmark_sample = [s for s in benchmark["data"] if s["id"] == sample["id"]][0]
    sample.update(benchmark_sample)

In [2]:
from collections import Counter
reasoning_types = Counter([s["reasoning_type"] for s in reasoning_types_results["data"] if "reasoning_type" in s])

reasoning_types

Counter({'creative': 2061, 'factual': 352})

In [3]:
from itertools import chain
creative_concepts = Counter(chain.from_iterable([s["concepts"] for s in reasoning_types_results["data"] if "concepts" in s]))

creative_concepts

Counter({'lateral thinking': 1740,
         'analogy': 830,
         'abstraction': 357,
         'joke': 226,
         'pun': 225,
         'metaphor': 207,
         'commonsense reasoning': 143,
         'poem': 83,
         'idiom': 79,
         'neologism': 36,
         'sarcasm': 34,
         'proverb': 31,
         'divergent thinking': 27,
         'compositionality': 13,
         'simile': 8,
         'historical reference': 5,
         'hyperbole': 3,
         'cultural reference': 2,
         'literary reference': 2,
         'reinterpretation': 1,
         'myth': 1,
         'irony': 1,
         'allegory': 1,
         'ambiguity': 1})

In [14]:
from statistics import mean
import random

factual_ids = [s["id"] for s in reasoning_types_results["data"] if "reasoning_type" in s and s["reasoning_type"] == "factual"]
creative_ids = [s["id"] for s in reasoning_types_results["data"] if "reasoning_type" in s and s["reasoning_type"] == "creative"]
factual_results = [r for r in model_results["data"] if r["id"] in factual_ids]
creative_results = [r for r in model_results["data"] if r["id"] in creative_ids]

min_len = min(len(factual_results), len(creative_results))
factual_results = random.sample(factual_results, min_len)
creative_results = random.sample(creative_results, min_len)

factual_acc = mean([int(r.get("gpt-4o_judge_match", False)) for r in factual_results])
creative_acc = mean([int(r.get("gpt-4o_judge_match", False)) for r in creative_results])

print(f"Number of factual samples: {len(factual_results)}")
print(f"Number of creative samples: {len(creative_results)}")
print(f"Factual accuracy: {factual_acc:.2%}")
print(f"Creative accuracy: {creative_acc:.2%}")

Number of factual samples: 352
Number of creative samples: 352
Factual accuracy: 54.26%
Creative accuracy: 34.94%


In [20]:
from collections import Counter

factual_difficulty_dist = Counter([result["difficulty_id"] for result in factual_results])

# factual_difficulty_dist

# visualize the distribution of difficulty types for factual samples
import seaborn as sns
import matplotlib.pyplot as plt

difficulties = ["Very Simple", "Simple", "Medium", "Hard", "Very Hard"]

difficulty_counts = [factual_difficulty_dist.get(i, 0) for i in range(1, 6)]

plt.figure(figsize=(8, 5))
ax = sns.barplot(x=difficulties, y=difficulty_counts, palette="Greens")
ax.set_xlabel("Difficulty", fontsize=12, color="#333333")
ax.set_ylabel("Count", fontsize=12, color="#333333")
plt.title("Distribution of Difficulty Types for Factual Samples", fontsize=14, color="#333333")
plt.savefig("../figures/factual-diff-dist.pdf", dpi=360, bbox_inches="tight", facecolor="white")
plt.close()

/var/folders/0x/3jp31_t54llb6607nqdym62c0000gn/T/ipykernel_5528/3486869672.py:16: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x=difficulties, y=difficulty_counts, palette="Greens")


In [21]:
from collections import Counter

creative_diff_dist = Counter([result["difficulty_id"] for result in creative_results])

# creative_diff_dist

# visualize the distribution of difficulty types for creative samples
import seaborn as sns
import matplotlib.pyplot as plt

difficulties = ["Very Simple", "Simple", "Medium", "Hard", "Very Hard"]

difficulty_counts = [creative_diff_dist.get(i, 0) for i in range(1, 6)]

plt.figure(figsize=(8, 5))
ax = sns.barplot(x=difficulties, y=difficulty_counts, palette="Greens")
ax.set_xlabel("Difficulty", fontsize=12, color="#333333")
ax.set_ylabel("Count", fontsize=12, color="#333333")
plt.title("Distribution of Difficulty Types for Creative Samples", fontsize=14, color="#333333")
plt.savefig("../figures/creative-diff-dist.pdf", dpi=360, bbox_inches="tight", facecolor="white")
plt.close()

/var/folders/0x/3jp31_t54llb6607nqdym62c0000gn/T/ipykernel_5528/959118794.py:16: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x=difficulties, y=difficulty_counts, palette="Greens")


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import squarify
 
# ── PASTE YOUR DATA HERE ──────────────────────────────────────────────────────
concept_counts = {c: val for c, val in creative_concepts.items() if val > 5}
concept_counts["other"] = sum(val for val in creative_concepts.values() if val <= 5)
# ─────────────────────────────────────────────────────────────────────────────
 
total = 2413
df = pd.DataFrame({
    "name":  list(concept_counts.keys()),
    "value": list(concept_counts.values()),
})
df["pct"] = df["value"] / total * 100
 
# ── Light pastel palette ──────────────────────────────────────────────────────
N      = len(df)
hues   = np.linspace(0, 1, N, endpoint=False)
colors = [mcolors.hsv_to_rgb([h, 0.30, 0.96]) for h in hues]
 
# ── Labels: concept name + count + percentage ─────────────────────────────────
# labels = [
#     f"{name}\n{value} ({pct:.1f}%)"
#     for name, value, pct in zip(df["name"], df["value"], df["pct"])
# ]

labels = [
    f"{name.replace(' ', chr(10)) if ' ' in name else name}\n{value} ({pct:.1f}%)" if "lateral" not in name else f"{name}\n{value} ({pct:.1f}%)"
    for name, value, pct in zip(df["name"], df["value"], df["pct"])
]
 
# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 10))
ax.set_axis_off()
 
squarify.plot(
    sizes=np.sqrt(df["value"]),
    label=labels,
    color=colors,
    text_kwargs={"color": "#333333", "fontsize": 16, "fontweight": "bold"},
    pad=True,
    ax=ax,
)
 
# plt.savefig("../figures/creative_concepts_dist.pdf",
#             dpi=360, bbox_inches="tight", facecolor="white")
plt.savefig("../figures/creative_concepts_dist.png",
            dpi=360, bbox_inches="tight", facecolor="white")
plt.close()
print("Done")

Done


In [23]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── PASTE YOUR DATA HERE ──────────────────────────────────────────────────────
num_concepts_dist = Counter([len(s["concepts"]) for s in reasoning_types_results["data"] if "concepts" in s])
# ─────────────────────────────────────────────────────────────────────────────

df = pd.DataFrame({
    "domains": list(num_concepts_dist.keys()),
    "count":   list(num_concepts_dist.values()),
})
df = df.sort_values("domains").reset_index(drop=True)
df["domains"] = df["domains"].astype(str)

# ── Light pastel palette (one color per bar) ──────────────────────────────────
N      = len(df)
hues   = np.linspace(0, 1, N, endpoint=False)
colors = [mcolors.hsv_to_rgb([h, 0.30, 0.96]) for h in hues]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6), facecolor="white")
ax.set_facecolor("white")

sns.barplot(data=df, x="domains", y="count", palette=colors, ax=ax)

# Count + percentage labels on top of each bar
total = df["count"].sum()
for bar in ax.patches:
    count = int(bar.get_height())
    pct = count / total * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + total * 0.003,
        f"{count}\n({pct:.1f}%)",
        ha="center", va="bottom", fontsize=20, fontweight="bold", color="#444444"
    )

ax.set_xlabel("Number of creative language constructs", fontsize=24, color="#444444")
ax.set_ylabel("Number of questions", fontsize=24, color="#444444")
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(colors="#666666")
ax.tick_params(axis='x', labelsize=18)
ax.tick_params(axis='y', labelsize=18)

plt.tight_layout()
plt.savefig("../figures/num_concepts_dist.pdf", dpi=360, bbox_inches="tight", facecolor="white")
plt.close()

/var/folders/0x/3jp31_t54llb6607nqdym62c0000gn/T/ipykernel_29116/1071364848.py:26: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df, x="domains", y="count", palette=colors, ax=ax)


In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# ── PASTE YOUR DATA HERE ──────────────────────────────────────────────────────
models = [
    ("Gemini-3.1-Pro (medium)", "../experiments/outputs/gemini-3.1-pro-preview/chgk_en_benchmark_eval_reasoning_model_en_s0_gemini-3.1-pro-preview_20260310_105704.json"),
    ("Gemini-3.1-Pro (high)", "../experiments/outputs/gemini-3.1-pro-preview/chgk_en_benchmark_eval_reasoning_model_en_s0_gemini-3.1-pro-preview_20260312_204451.json"),
    ("GPT-4.1", "../experiments/outputs/gpt-4.1/chgk_en_benchmark_eval_cot_answer_en_s0_gpt-4.1_20260228_123352.json"),
    ("Qwen3-235B-A22B-Thinking", "../experiments/outputs/qwen3-235b-a22b-thinking-2507/chgk_en_benchmark_eval_reasoning_model_en_s0_qwen3-235b-a22b-thinking-2507_20260312_203137.json"),
    ("DeepSeek-V3.2", "../experiments/outputs/deepseek-v3.2/chgk_en_benchmark_eval_reasoning_model_en_s0_deepseek-v3.2_20260315_133331.json"),
    ("GPT-5.4 (medium)", "../experiments/outputs/gpt-5.4/chgk_en_benchmark_eval_reasoning_model_en_s0_gpt-5.4_20260310_110106.json"),
]
results = {}
concepts = [concept for concept, counts in creative_concepts.items() if counts > 30]
concepts = sorted(concepts, key=lambda c: creative_concepts[c], reverse=True)

for model, model_path in models:
    model_results = read_json(model_path)
    model_accs = []
    for concept in concepts:
        concept_sample_ids = [s["id"] for s in reasoning_types_results["data"] if concept in s.get("concepts", [])]
        concept_samples = [s for s in model_results["data"] if s["id"] in concept_sample_ids]
        concept_acc = np.mean([int(s.get("gpt-4o_judge_match", False)) for s in concept_samples])
        model_accs.append(concept_acc)
    results[model] = model_accs
# ─────────────────────────────────────────────────────────────────────────────

df = pd.DataFrame(results, index=[f"{concept} ({creative_concepts[concept]})" for concept in concepts]).T

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, len(results)), facecolor="white")
cmap = sns.light_palette("#7a5c3a", as_cmap=True)  # cream → warm brown

sns.heatmap(
    df,
    ax=ax,
    cmap=cmap,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 9},
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Accuracy", "shrink": 0.6},
    vmin=0, vmax=1,
)

ax.set_xlabel("Concept", fontsize=12, color="#444444")
ax.set_ylabel("Model", fontsize=12, color="#444444")
ax.tick_params(axis="x", labelsize=11, colors="#444444", rotation=70)
ax.tick_params(axis="y", labelsize=9,  colors="#444444", rotation=0)

plt.tight_layout()
plt.savefig("../figures/creative-results.pdf", dpi=360, bbox_inches="tight", facecolor="white")
plt.close()

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# ── PASTE YOUR DATA HERE ──────────────────────────────────────────────────────

models = [
    ("Gemini-3.1-Pro (medium)", "../experiments/outputs/gemini-3.1-pro-preview/chgk_ru_benchmark_eval_reasoning_model_ru_en_s0_gemini-3.1-pro-preview_20260310_160621.json"),
    ("Gemini-3.1-Pro (high)", "../experiments/outputs/gemini-3.1-pro-preview/chgk_ru_benchmark_eval_reasoning_model_ru_en_s0_gemini-3.1-pro-preview_20260313_232947.json"),
    ("GPT-4.1", "../experiments/outputs/gpt-4.1/chgk_ru_benchmark_eval_cot_answer_ru_en_s0_gpt-4.1_20260228_160027.json"),
    ("Qwen3-235B-A22B-Thinking", "../experiments/outputs/qwen3-235b-a22b-thinking-2507/chgk_ru_benchmark_eval_reasoning_model_ru_en_s0_qwen3-235b-a22b-thinking-2507_20260314_102839.json"),
    ("DeepSeek-V3.2", "../experiments/outputs/deepseek-v3.2/chgk_ru_benchmark_eval_reasoning_model_ru_en_s0_deepseek-v3.2_20260315_165350.json"),
    ("GPT-5.4 (medium)", "../experiments/outputs/gpt-5.4/chgk_ru_benchmark_eval_reasoning_model_ru_en_s0_gpt-5.4_20260310_132133.json"),
]
results = {}
concepts = [concept for concept, counts in creative_concepts.items() if counts > 30]
concepts = sorted(concepts, key=lambda c: creative_concepts[c], reverse=True)

for model, model_path in models:
    model_results = read_json(model_path)
    model_accs = []
    for concept in concepts:
        concept_sample_ids = [s["id"] for s in reasoning_types_results["data"] if concept in s.get("concepts", [])]
        concept_samples = [s for s in model_results["data"] if s["id"] in concept_sample_ids]
        concept_acc = np.mean([int(s.get("gpt-4o_judge_match", False)) for s in concept_samples])
        model_accs.append(concept_acc)
    results[model] = model_accs
# ─────────────────────────────────────────────────────────────────────────────

df = pd.DataFrame(results, index=[f"{concept} ({creative_concepts[concept]})" for concept in concepts]).T

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, len(results)), facecolor="white")
cmap = sns.light_palette("#7a5c3a", as_cmap=True)  # cream → warm brown

sns.heatmap(
    df,
    ax=ax,
    cmap=cmap,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 9},
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Accuracy", "shrink": 0.6},
    vmin=0, vmax=1,
)

ax.set_xlabel("Concept", fontsize=12, color="#444444")
ax.set_ylabel("Model", fontsize=12, color="#444444")
ax.tick_params(axis="x", labelsize=11, colors="#444444", rotation=70)
ax.tick_params(axis="y", labelsize=9,  colors="#444444", rotation=0)

plt.tight_layout()
plt.savefig("../figures/creative-results-ru.pdf", dpi=360, bbox_inches="tight", facecolor="white")
plt.close()

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from cresowlve.utils import read_json

# ─────────────────────────────────────────────────────────────────────────────

creative_ids = [s["id"] for s in reasoning_types_results["data"] if "reasoning_type" in s and s["reasoning_type"] == "creative"]
factual_ids = [s["id"] for s in reasoning_types_results["data"] if "reasoning_type" in s and s["reasoning_type"] == "factual"]

def plot_creative_vs_factual(model_results, metric="exact_match"):
    path_specs = [
        ("en_path", "CresOWLve-En"),
        ("ru_path", "CresOWLve-Ru"),
    ]

    earthy_palette = [
        "#7A5C3E",
        "#6B8E23",
        "#A97142",
        "#8B6F47",
        "#5E7D6A",
        "#B08968",
        "#4F6D5A",
        "#9C6644",
        "#A3B18A",
        "#6C584C",
    ]

    path_frames = {}
    for path_key, path_label in path_specs:
        rows = []
        for result in model_results:
            if not result.get(path_key):
                continue

            outputs = read_json(result[path_key])
            model_key = f"{result['model_name']}"
            thinking = result.get("thinking")
            if thinking:
                model_key += f" ({thinking})"

            creative_results = [r for r in outputs["data"] if r["id"] in creative_ids]
            factual_results = [r for r in outputs["data"] if r["id"] in factual_ids]
            sample_size = min(len(creative_results), len(factual_results))

            if sample_size == 0:
                continue

            creative_accs = []
            factual_accs = []

            for _ in range(1000):
                creative_sample = np.random.choice(creative_results, sample_size, replace=True)
                factual_sample = np.random.choice(factual_results, sample_size, replace=True)

                creative_acc = np.mean([int(r.get(metric, False)) for r in creative_sample])
                factual_acc = np.mean([int(r.get(metric, False)) for r in factual_sample])

                creative_accs.append(creative_acc)
                factual_accs.append(factual_acc)

            for score in factual_accs:
                rows.append({"Model": model_key, "Category": "factual", "Accuracy": score})
            for score in creative_accs:
                rows.append({"Model": model_key, "Category": "creative", "Accuracy": score})

        df = pd.DataFrame(rows)
        if not df.empty:
            df["Category"] = pd.Categorical(df["Category"], categories=["factual", "creative"], ordered=True)
        path_frames[path_label] = df

    fig, axes = plt.subplots(1, len(path_specs), figsize=(15, 6), sharey=True, facecolor="#FAF8F3")
    if len(path_specs) == 1:
        axes = [axes]

    markers = ["o", "s", "^", "D", "P", "X", "*", "v", "<", ">"]
    legend_handles = None
    legend_labels = None

    for ax, (_, path_label) in zip(axes, path_specs):
        df = path_frames[path_label]
        ax.set_facecolor("#FCFBF7")

        if df.empty:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", fontsize=12, color="#4D463F")
            ax.set_title(path_label, fontsize=14, color="#3E3A36")
            ax.set_xlabel("Category", fontsize=12, color="#4D463F")
            continue

        point = sns.pointplot(
            data=df,
            x="Category",
            y="Accuracy",
            hue="Model",
            dodge=0.4,
            markers=markers[:len(df["Model"].unique())],
            linestyles="-",
            errorbar="sd",
            capsize=0.05,
            markersize=8,
            linewidth=2,
            err_kws={"linewidth": 1.5, "color": "#8C816F"},
            palette=earthy_palette,
            ax=ax,
        )

        if legend_handles is None:
            legend_handles, legend_labels = point.get_legend_handles_labels()

        if ax.get_legend() is not None:
            ax.get_legend().remove()

        ax.set_title(path_label, fontsize=14, color="#3E3A36")
        ax.set_xlabel("Category", fontsize=12, color="#4D463F")
        ax.grid(True, axis="y", color="#D9D3C7", linestyle="-", linewidth=0.8, alpha=0.8)
        ax.grid(True, axis="x", color="#E8E3DA", linestyle=":", linewidth=0.7, alpha=0.6)
        ax.tick_params(labelsize=11, colors="#3E3A36")
        ax.spines[["top", "right"]].set_visible(False)
        ax.spines["left"].set_color("#B8AD98")
        ax.spines["bottom"].set_color("#B8AD98")
        ax.set_ylim(0, 1)

    axes[0].set_ylabel("Accuracy", fontsize=12, color="#4D463F")
    for ax in axes[1:]:
        ax.set_ylabel("")

    if legend_handles and legend_labels:
        legend = fig.legend(
            legend_handles,
            legend_labels,
            loc="lower center",
            bbox_to_anchor=(0.5, -0.03),
            ncol=min(4, len(legend_labels)),
            frameon=True,
            framealpha=0.95,
            fontsize=10,
            title="Model",
            title_fontsize=11,
        )
        legend.get_frame().set_facecolor("#F3EFE6")
        legend.get_frame().set_edgecolor("#CFC6B4")

    plt.tight_layout(rect=[0, 0.08, 1, 1])
    plt.savefig(f"../figures/creative-vs-factual-{metric}.pdf", dpi=360, bbox_inches="tight", facecolor="white")
    plt.close()

results = read_json("results.json")

sub_results = [
    result for result in results
    if result["model_name"] in [
        "Llama-3.3-70B-Instruct",
        "Mistral-Large-3-675B-Instruct",
        "GPT-4.1",
        "Qwen3-235B-A22B-Thinking",
        "DeepSeek-V3.2",
        "GPT-5.4",
        "Gemini-3.1-Pro",
    ] and result["thinking"] in [None, "medium", "adaptive", "high"]
]

plot_creative_vs_factual(sub_results, metric="exact_match")
plot_creative_vs_factual(sub_results, metric="gpt-4o_judge_match")

/var/folders/0x/3jp31_t54llb6607nqdym62c0000gn/T/ipykernel_5528/2105442207.py:92: UserWarning: The palette list has more values (10) than needed (8), which may not be intended.
  point = sns.pointplot(
/var/folders/0x/3jp31_t54llb6607nqdym62c0000gn/T/ipykernel_5528/2105442207.py:92: UserWarning: The palette list has more values (10) than needed (8), which may not be intended.
  point = sns.pointplot(
/var/folders/0x/3jp31_t54llb6607nqdym62c0000gn/T/ipykernel_5528/2105442207.py:92: UserWarning: The palette list has more values (10) than needed (8), which may not be intended.
  point = sns.pointplot(
/var/folders/0x/3jp31_t54llb6607nqdym62c0000gn/T/ipykernel_5528/2105442207.py:92: UserWarning: The palette list has more values (10) than needed (8), which may not be intended.
  point = sns.pointplot(


In [8]:
# print LLM judge match difference between creative and factual for each model
for result in sub_results:
    if not result.get("en_path"):
        continue

    outputs = read_json(result["en_path"])
    model_key = f"{result['model_name']}"
    thinking = result.get("thinking")
    if thinking:
        model_key += f" ({thinking})"

    creative_results = [r for r in outputs["data"] if r["id"] in creative_ids]
    factual_results = [r for r in outputs["data"] if r["id"] in factual_ids]
    sample_size = min(len(creative_results), len(factual_results))

    if sample_size == 0:
        continue

    creative_accs = []
    factual_accs = []

    for _ in range(1000):
        creative_sample = np.random.choice(creative_results, sample_size, replace=True)
        factual_sample = np.random.choice(factual_results, sample_size, replace=True)

        creative_acc = np.mean([int(r.get("gpt-4o_judge_match", False)) for r in creative_sample])
        factual_acc = np.mean([int(r.get("gpt-4o_judge_match", False)) for r in factual_sample])

        creative_accs.append(creative_acc)
        factual_accs.append(factual_acc)

    acc_diff = np.mean(creative_accs) - np.mean(factual_accs)
    print(f"{model_key}: Creative vs Factual GPT-4o Judge Match Difference: {acc_diff:.2%}")

Llama-3.3-70B-Instruct: Creative vs Factual GPT-4o Judge Match Difference: -16.89%
Mistral-Large-3-675B-Instruct: Creative vs Factual GPT-4o Judge Match Difference: -17.21%
GPT-4.1: Creative vs Factual GPT-4o Judge Match Difference: -12.23%
Qwen3-235B-A22B-Thinking (adaptive): Creative vs Factual GPT-4o Judge Match Difference: -15.51%
DeepSeek-V3.2 (adaptive): Creative vs Factual GPT-4o Judge Match Difference: -16.89%
GPT-5.4 (medium): Creative vs Factual GPT-4o Judge Match Difference: -12.39%
Gemini-3.1-Pro (medium): Creative vs Factual GPT-4o Judge Match Difference: -9.96%
Gemini-3.1-Pro (high): Creative vs Factual GPT-4o Judge Match Difference: -6.08%


In [9]:
# print LLM judge match difference between creative and factual for each model
for result in sub_results:
    if not result.get("ru_path"):
        continue

    outputs = read_json(result["ru_path"])
    model_key = f"{result['model_name']}"
    thinking = result.get("thinking")
    if thinking:
        model_key += f" ({thinking})"

    creative_results = [r for r in outputs["data"] if r["id"] in creative_ids]
    factual_results = [r for r in outputs["data"] if r["id"] in factual_ids]
    sample_size = min(len(creative_results), len(factual_results))

    if sample_size == 0:
        continue

    creative_accs = []
    factual_accs = []

    for _ in range(1000):
        creative_sample = np.random.choice(creative_results, sample_size, replace=True)
        factual_sample = np.random.choice(factual_results, sample_size, replace=True)

        creative_acc = np.mean([int(r.get("gpt-4o_judge_match", False)) for r in creative_sample])
        factual_acc = np.mean([int(r.get("gpt-4o_judge_match", False)) for r in factual_sample])

        creative_accs.append(creative_acc)
        factual_accs.append(factual_acc)

    acc_diff = np.mean(creative_accs) - np.mean(factual_accs)
    print(f"{model_key}: Creative vs Factual GPT-4o Judge Match Difference: {acc_diff:.2%}")

Llama-3.3-70B-Instruct: Creative vs Factual GPT-4o Judge Match Difference: -12.24%
Mistral-Large-3-675B-Instruct: Creative vs Factual GPT-4o Judge Match Difference: -12.67%
GPT-4.1: Creative vs Factual GPT-4o Judge Match Difference: -16.00%
Qwen3-235B-A22B-Thinking (adaptive): Creative vs Factual GPT-4o Judge Match Difference: -10.50%
DeepSeek-V3.2 (adaptive): Creative vs Factual GPT-4o Judge Match Difference: -13.38%
GPT-5.4 (medium): Creative vs Factual GPT-4o Judge Match Difference: -5.88%
Gemini-3.1-Pro (medium): Creative vs Factual GPT-4o Judge Match Difference: -2.73%
Gemini-3.1-Pro (high): Creative vs Factual GPT-4o Judge Match Difference: -2.14%
